In [ ]:
!pip install -q transformers datasets rouge-score bert_score nltk -q

In [ ]:
!pip install langchain_community langchain-experimental langchain-openai -q

In [3]:
import os
import json
import re
import math
import time

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
import bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

In [10]:
# ============================================================
# CONFIG
# ============================================================

# --- Local model to evaluate ---
MODEL_NAME = "t5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_GEN_LENGTH = 100

# --- Judge LLM (OpenRouter or NVIDIA NIM) ---
JUDGE_PROVIDER = "openrouter"  # "openrouter" or "nvidia"

JUDGE_CONFIGS = {
    "openrouter": {
        "api_base": "https://openrouter.ai/api/v1",
        "api_key": "api",   # <-- replace
        "model": "nvidia/nemotron-3-nano-30b-a3b:free",  # or any OpenRouter model
    },
    "nvidia": {
        "api_base": "https://integrate.api.nvidia.com/v1",
        "api_key": "your-nvidia-api-key",       # <-- replace
        "model": "model",
    },
}

JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_RETRIES = 2
JUDGE_SLEEP_BETWEEN_CALLS = 0.5  # seconds, be polite to rate limits

In [5]:

# ============================================================
# DATASET
# ============================================================

dataset = [
    {
        "article": "The Eiffel Tower is located in Paris and was completed in 1889. It is a major tourist attraction.",
        "highlights": "The Eiffel Tower in Paris was completed in 1889."
    },
    {
        "article": "Python is a programming language known for its simplicity and readability. It is used widely in AI.",
        "highlights": "Python is a simple, readable language popular in AI."
    },
    {
        "article": "The Amazon rainforest is the largest tropical rainforest in the world and is home to diverse wildlife.",
        "highlights": "Amazon rainforest is the world's largest and rich in biodiversity."
    }
]


In [6]:
# ============================================================
# STEP 1: LOAD MODEL & GENERATE PREDICTIONS
# ============================================================

def load_model(model_name=MODEL_NAME, device=DEVICE):
    print(f"Loading model '{model_name}' on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    model.eval()
    return tokenizer, model


def generate_predictions(dataset, tokenizer, model, device=DEVICE, max_length=MAX_GEN_LENGTH):
    predictions, references, losses = [], [], []

    for sample in dataset:
        input_text = "summarize: " + sample["article"]
        input_ids = tokenizer(input_text, return_tensors="pt", truncation=True).input_ids.to(device)

        with torch.no_grad():
            output_ids = model.generate(input_ids, max_length=max_length)
            pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)

            target_ids = tokenizer(sample["highlights"], return_tensors="pt", truncation=True).input_ids.to(device)
            loss = model(input_ids=input_ids, labels=target_ids).loss.item()

        predictions.append(pred)
        references.append(sample["highlights"])
        losses.append(loss)

    return predictions, references, losses



In [7]:
# ============================================================
# STEP 2: CLASSIC METRICS (BLEU, ROUGE, Perplexity, BERTScore)
# ============================================================

def compute_bleu(predictions, references):
    def real_bleu(pred, ref):
        pred_tokens = pred.lower().split()
        ref_tokens = [ref.lower().split()]
        smoothie = SmoothingFunction().method4
        return sentence_bleu(ref_tokens, pred_tokens, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    scores = [real_bleu(pred, ref) for pred, ref in zip(predictions, references)]
    return scores, sum(scores) / len(scores)


def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge1, rouge2, rougeL = [], [], []

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1.append(scores["rouge1"].fmeasure)
        rouge2.append(scores["rouge2"].fmeasure)
        rougeL.append(scores["rougeL"].fmeasure)

    return {
        "rouge1": (rouge1, sum(rouge1) / len(rouge1)),
        "rouge2": (rouge2, sum(rouge2) / len(rouge2)),
        "rougeL": (rougeL, sum(rougeL) / len(rougeL)),
    }


def compute_perplexity(losses):
    avg_loss = sum(losses) / len(losses)
    return math.exp(avg_loss)


def compute_bertscore(predictions, references):
    P, R, F1 = bert_score.score(predictions, references, lang="en", verbose=False)
    return {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item(),
    }


In [8]:
# ============================================================
# STEP 3: LLM-AS-JUDGE
# ============================================================

JUDGE_SYSTEM_PROMPT = """You are an expert evaluator of text summarization quality.
You will be given a source article, a reference (gold) summary, and a candidate
summary generated by a model. Score the candidate summary on the following
criteria, each from 1 (very poor) to 5 (excellent):

- relevance: Does it capture the key points of the article?
- coherence: Is it well-structured and logically organized?
- consistency: Is it factually consistent with the article (no hallucinations)?
- fluency: Is it grammatically correct and natural to read?

Respond with ONLY a JSON object in this exact format, no other text:
{"relevance": <int>, "coherence": <int>, "consistency": <int>, "fluency": <int>, "reasoning": "<one sentence justification>"}
"""

JUDGE_USER_TEMPLATE = """Article:
{article}

Reference summary:
{reference}

Candidate summary:
{candidate}

Provide your scores as JSON."""


def build_judge_llm(provider=JUDGE_PROVIDER):
    cfg = JUDGE_CONFIGS[provider]
    os.environ["OPENAI_API_KEY"] = cfg["api_key"]
    os.environ["OPENAI_API_BASE"] = cfg["api_base"]

    return ChatOpenAI(
        model=cfg["model"],
        temperature=JUDGE_TEMPERATURE,
        openai_api_base=cfg["api_base"],
        openai_api_key=cfg["api_key"],
        request_timeout=60,
    )


def parse_judge_output(raw_text):
    """Robustly extract JSON from the judge's response."""
    text = raw_text.strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
        return None


def llm_judge_score(article, reference, candidate, llm, max_retries=JUDGE_MAX_RETRIES):
    messages = [
        SystemMessage(content=JUDGE_SYSTEM_PROMPT),
        HumanMessage(content=JUDGE_USER_TEMPLATE.format(
            article=article, reference=reference, candidate=candidate
        )),
    ]

    for attempt in range(max_retries + 1):
        try:
            response = llm.invoke(messages)
            parsed = parse_judge_output(response.content)
            if parsed and all(k in parsed for k in ["relevance", "coherence", "consistency", "fluency"]):
                return parsed
        except Exception as e:
            print(f"  Judge call failed (attempt {attempt + 1}): {e}")

    return {"relevance": None, "coherence": None, "consistency": None,
             "fluency": None, "reasoning": "PARSE_FAILED"}


def run_llm_judge(dataset, predictions, llm):
    results = []
    for i, (sample, pred) in enumerate(zip(dataset, predictions)):
        print(f"  Judging sample {i + 1}/{len(dataset)}...")
        result = llm_judge_score(
            article=sample["article"],
            reference=sample["highlights"],
            candidate=pred,
            llm=llm,
        )
        results.append(result)
        time.sleep(JUDGE_SLEEP_BETWEEN_CALLS)
    return results


def safe_avg(results, key):
    vals = [r[key] for r in results if r.get(key) is not None]
    return sum(vals) / len(vals) if vals else float("nan")

In [ ]:
# ============================================================
# MAIN
# ============================================================

def main():
    # --- Generate predictions from local model ---
    tokenizer, model = load_model()
    predictions, references, losses = generate_predictions(dataset, tokenizer, model)

    print("\n=== Predictions ===")
    for i, (pred, ref) in enumerate(zip(predictions, references)):
        print(f"[{i+1}] Pred: {pred}")
        print(f"    Ref:  {ref}")

    # --- Classic metrics ---
    print("\n=== Classic Metrics ===")
    _, avg_bleu = compute_bleu(predictions, references)
    print(f"BLEU Score: {avg_bleu:.4f}")

    rouge_results = compute_rouge(predictions, references)
    print(f"ROUGE-1 F1 Score: {rouge_results['rouge1'][1]:.4f}")
    print(f"ROUGE-2 F1 Score: {rouge_results['rouge2'][1]:.4f}")
    print(f"ROUGE-L F1 Score: {rouge_results['rougeL'][1]:.4f}")

    perplexity = compute_perplexity(losses)
    print(f"Perplexity: {perplexity:.2f}")

    bert_results = compute_bertscore(predictions, references)
    print(f"BERTScore - Precision: {bert_results['precision']:.4f}")
    print(f"BERTScore - Recall: {bert_results['recall']:.4f}")
    print(f"BERTScore - F1: {bert_results['f1']:.4f}")

    # --- LLM-as-judge ---
    print("\n=== LLM-as-Judge ===")
    judge_llm = build_judge_llm()
    judge_results = run_llm_judge(dataset, predictions, judge_llm)

    print(f"LLM Judge - Relevance:   {safe_avg(judge_results, 'relevance'):.2f}/5")
    print(f"LLM Judge - Coherence:   {safe_avg(judge_results, 'coherence'):.2f}/5")
    print(f"LLM Judge - Consistency: {safe_avg(judge_results, 'consistency'):.2f}/5")
    print(f"LLM Judge - Fluency:     {safe_avg(judge_results, 'fluency'):.2f}/5")

    overall = sum([safe_avg(judge_results, k) for k in
                    ["relevance", "coherence", "consistency", "fluency"]]) / 4
    print(f"LLM Judge - Overall:     {overall:.2f}/5")

    print("\n--- Per-sample judge reasoning ---")
    for i, r in enumerate(judge_results):
        print(f"[{i+1}] {r}")

    # --- Final summary ---
    print("\n=== FINAL SUMMARY ===")
    summary = {
        "bleu": avg_bleu,
        "rouge1": rouge_results['rouge1'][1],
        "rouge2": rouge_results['rouge2'][1],
        "rougeL": rouge_results['rougeL'][1],
        "perplexity": perplexity,
        "bertscore_precision": bert_results['precision'],
        "bertscore_recall": bert_results['recall'],
        "bertscore_f1": bert_results['f1'],
        "llm_judge_relevance": safe_avg(judge_results, 'relevance'),
        "llm_judge_coherence": safe_avg(judge_results, 'coherence'),
        "llm_judge_consistency": safe_avg(judge_results, 'consistency'),
        "llm_judge_fluency": safe_avg(judge_results, 'fluency'),
        "llm_judge_overall": overall,
    }
    print(json.dumps(summary, indent=2))

    return summary


if __name__ == "__main__":
    main()